# Debugging Missings Pre & Post County Formations Estimation Merge

## Setup

In [21]:
# Packages
import pandas as pd

# Loading Datasets
state_forms = pd.read_csv('../../data_clean/annualized_state_formations.csv',
                           dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
state_apps = pd.read_csv('../../data_clean/state_apps_annual.csv',
                         dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
county_apps = pd.read_csv('../../data_clean/annual_county_applications.csv',
                          dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})

## Pre-Merge Missing Data
Getting dataframes of missing observations from all three datasets used in the calculation PRE-MERGE to compare with the missing observations in the POST-MERGE dataset to determine true missing vs merge issue.

In [22]:
# State Formations
sf_og_missing = state_forms[state_forms['state_formations'].isna()]
sf_og_missing.to_csv('sf_og_missing.csv') 

# State Applications
sa_og_missing = state_apps[state_apps['state_apps'].isna()]
sa_og_missing.to_csv('sa_og_missing.csv')

# County Applications
ca_og_missing = county_apps[county_apps['county_apps'].isna()]
ca_og_missing.to_csv('ca_og_missing.csv')

## Updated county_formations_estimation.py file
All actual code edits happen in the .py file. I copy-paste the updated file (minus the setup section) here and comment out the final line. 

In [23]:

# #################################
# MERGING DATASETS

# Merging state formations and state applications
state = pd.merge(state_forms, state_apps, on=['year', 'state_fips', 'STATE', 'STATE_NAME'])
assert state.groupby(['state_fips', 'year']).size().eq(1).all()

# Merging state dataframe with county applications
est = pd.merge(county_apps, state, on=['year', 'state_fips', 'STATE'], validate='many_to_one')
assert est.groupby(['full_fips', 'year']).size().eq(1).all()

# Merge validation
assert est['state_apps'].notna().all()

# Columns Cleanup
est = est.rename(columns={'County': 'county'})
est = est.reindex(columns = ['STATE', 'year', 'county', 'full_fips', 'state_formations', 'state_apps', 'county_apps', 'state_fips', 'county_fips', 'STATE_NAME'])


# #################################
# ESTIMATING COUNTY FORMATIONS

# Correcting variable types
est['county_apps'] = pd.to_numeric(est['county_apps'], errors='coerce')

# Calculating estimated county formations
est['county_forms_est'] = (est['state_formations']/est['state_apps']) * est['county_apps']


# #################################
# SAVING DATASET
# est.to_csv('data_clean/county_formations_estimation.csv', index=False)

## Investigation Code:
Debugging shenanigans start here!

Abbreviations Guide:
* sf = state formations
* sa = state applications
* ca = county applications
* og = original, meaning pre-merge data

In [24]:
# FINDING POST-MERGE MISSING DATA IN EACH VARIABLE
sf_missing = est[est['state_formations'].isna()] # Missings in state formations
sa_missing = est[est['state_apps'].isna()] # Missings in state applications
ca_missing = est[est['county_apps'].isna()] # Missings in county applications

In [28]:
# Merging sf missing dataframes to separate out true missing (47) vs the large remainder of missings in the final data
sf_pre_post_missing = pd.merge(sf_missing, sf_og_missing, on=['state_fips', 'year'], indicator=True)

sf_missing_problematic = sf_pre_post_missing[sf_pre_post_missing['_merge']!= 'both']

sf_missing.groupby('state_fips').size().sort_values(ascending=False)

state_fips
46    402
38    318
54    220
02    160
50     98
56     92
35     33
23     32
44     30
33     10
15      5
11      4
dtype: int64